# Lesson 06 Lab — Global Sparsity and Layer-wise Budget Allocation

**Puzzle:** Should a fixed 50% global budget prune every layer by 50%?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

Layers transform different signals and have different redundancy. A global threshold spends zeros where weights are small, while a uniform per-layer target ignores sensitivity. A budget table should preserve the total constraint and show why protected or aggressive allocations were assigned.


## 0. Predict before running

1. Predict which layer will be protected by the calibration sweep.
2. Predict whether uniform and global magnitude masks use the same per-layer rates.
3. Name the quality metric and total-budget invariant required for fairness.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

A three-layer MLP, calibration inputs, one global magnitude mask, a uniform 50% mask, and a sensitivity-aware allocation are compared at equal total nonzero count using held-out output reconstruction.

- Global sparsity is a constraint across tensors, not a uniform policy.
- Layer sensitivity must be measured on representative inputs.
- Budget comparisons require equal total nonzeros.


## 2. Derive the mechanism

For network output `f(x; W)`, a layer's pruning cost depends on downstream amplification and the input distribution, not only its weight histogram. A first-order sensitivity sweep can mask a small fraction in one layer at a time and measure output change. Budgets can then be allocated inversely to observed sensitivity while solving the global nonzero constraint. The experiment keeps total zeros equal so quality differences come from allocation rather than extra capacity.

Keep value sparsity, physical shape, representation, and runtime evidence separate.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 6
LESSON_TITLE = 'Global Sparsity and Layer-wise Budget Allocation'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260814
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | uniform 50% magnitude pruning in every layer |
| Candidate | global thresholding and sensitivity-aware per-layer allocation |
| Held constant | dense weights, calibration/held-out tensors, global zero count, dtype, and seed |
| Measurements | per-layer sparsity, total sparsity, held-out RMSE, cosine similarity, and calibration sensitivity |
| Evidence | `numerical-model` |

**Experiment:** Compare uniform, global-magnitude, and sensitivity-aware masks at the same 50% global zero budget.


## 5. Read the experiment code

The notebook first perturbs each layer separately to obtain a small calibration sensitivity score. It then constructs three cloned models and checks exact total sparsity before measuring held-out output error. The allocation heuristic is intentionally simple; the evidence target is the budget principle, not a claim of optimal pruning.

Do not execute until the code implements the frozen table above.


In [2]:
torch.manual_seed(SEED)
base = nn.Sequential(nn.Linear(64, 128, bias=False), nn.ReLU(), nn.Linear(128, 64, bias=False), nn.ReLU(), nn.Linear(64, 32, bias=False)).to(DEVICE).eval()
cal = torch.randn(256, 64, device=DEVICE)
held = torch.randn(256, 64, device=DEVICE) * 1.2
with torch.inference_mode(): ref_cal, ref_held = base(cal), base(held)
names = [name for name, p in base.named_parameters() if p.ndim == 2]

def clone_with_masks(mask_by_name):
    model = copy.deepcopy(base)
    with torch.no_grad():
        for name, p in model.named_parameters():
            if name in mask_by_name: p.mul_(mask_by_name[name])
    return model

sensitivities = {}
for name, p in base.named_parameters():
    if name in names:
        model = clone_with_masks({name: magnitude_mask(p, 0.30)})
        with torch.inference_mode(): sensitivities[name] = tensor_metrics(ref_cal, model(cal))["rmse"]

uniform_masks = {name: magnitude_mask(dict(base.named_parameters())[name], 0.50) for name in names}
all_values = torch.cat([dict(base.named_parameters())[name].detach().abs().flatten() for name in names])
k = int(all_values.numel() * 0.50); threshold = torch.topk(all_values, k, largest=False).values.max()
global_masks_dict = {name: (dict(base.named_parameters())[name].detach().abs() > threshold).float() for name in names}
adjusted_parts = []
for name in names:
    adjusted_parts.append((dict(base.named_parameters())[name].detach().abs() * (sensitivities[name] + 1e-6)).flatten())
adjusted = torch.cat(adjusted_parts); ath = torch.topk(adjusted, k, largest=False).values.max()
aware_masks = {name: ((dict(base.named_parameters())[name].detach().abs() * (sensitivities[name] + 1e-6)) > ath).float() for name in names}

uniform = clone_with_masks(uniform_masks); global_model = clone_with_masks(global_masks_dict); aware = clone_with_masks(aware_masks)
with torch.inference_mode():
    um = tensor_metrics(ref_held, uniform(held)); gm = tensor_metrics(ref_held, global_model(held)); am = tensor_metrics(ref_held, aware(held))
total_sparsity = sum((m == 0).sum().item() for m in aware_masks.values()) / sum(m.numel() for m in aware_masks.values())
metrics = {
    "uniform_rmse": um["rmse"], "global_rmse": gm["rmse"], "aware_rmse": am["rmse"],
    "uniform_cosine": um["cosine"], "global_cosine": gm["cosine"], "aware_cosine": am["cosine"],
    "total_sparsity": total_sparsity,
    "most_sensitive_layer": max(sensitivities, key=sensitivities.get),
    "sensitivities": sensitivities,
    "aware_layer_sparsity": {name: zero_fraction(mask) for name, mask in aware_masks.items()},
}
analysis = (
    f"All candidates used approximately {total_sparsity:.1%} global sparsity. Held-out RMSE was "
    f"{um['rmse']:.6f} for uniform, {gm['rmse']:.6f} for global magnitude, and {am['rmse']:.6f} "
    f"for the sensitivity-adjusted allocation. The calibration sweep identified "
    f"`{metrics['most_sensitive_layer']}` as most sensitive. This validates the budget experiment, not optimality of the heuristic."
)


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Uniform RMSE | 0.067273 |
| Global RMSE | 0.062327 |
| Aware RMSE | 0.061447 |
| Total sparsity | 50.00% |
| Most sensitive layer | 4.weight |


## 7. Interpret rather than merely print

All candidates used approximately 50.0% global sparsity. Held-out RMSE was 0.067273 for uniform, 0.062327 for global magnitude, and 0.061447 for the sensitivity-adjusted allocation. The calibration sweep identified `4.weight` as most sensitive. This validates the budget experiment, not optimality of the heuristic.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. The CUDA experiment isolates a numerical mechanism. It is not a full paper reproduction, trained production model, or native sparse-kernel benchmark.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 6,
    "title": 'Global Sparsity and Layer-wise Budget Allocation',
    "environment": ENV,
    "evidence_label": 'numerical-model',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'A global target needs a measured allocation rule; uniform layer rates are merely one candidate.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 6,
  "title": "Global Sparsity and Layer-wise Budget Allocation",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260814
  },
  "evidence_label": "numerical-model",
  "metrics": {
    "uniform_rmse": 0.06727341562509537,
    "global_rmse": 0.06232653558254242,
    "aware_rmse": 0.06144710257649422,
    "uniform_cosine": 0.8347758054733276,
    "global_cosine": 0.8603799939155579,
    "aware_cosine": 0.8644721508026123,
    "total_sparsity": 0.5,
    "most_sensitive_layer": "4.weight",
    "sensitivities": {
      "0.weight": 0.015526210889220238,
      "2.weight": 0.015088478103280067,
      "4.weight": 0.016306765377521515
    },
    "aware_layer_sparsity": {
      "0.weight": 0.41796875,
      "2.weight": 0.611083984375,
      "4.weight": 0.3837890625
    }
  },
  "analysis": "All candidates used approximately 50.0% global sparsity. Held-o

## 9. Make the bounded decision

> A global target needs a measured allocation rule; uniform layer rates are merely one candidate.

**Acceptance/rollback:** Accept a layer budget only when the total constraint is exact and the ranking remains stable on held-out data or multiple calibration slices.

**Failure analysis:** Using the held-out set to allocate budgets leaks evaluation. Very small calibration batches make sensitivity noisy, and equal zero counts do not ensure equal metadata or runtime cost across layers. Hardware-aware costs may need a different budget unit than parameters.


## 10. Extend the evidence

Repeat the sweep across domains, optimize budgets in latency-weighted channel units, and test whether protected early layers remain protected after recovery training.

The full evidence boundary and references are in [`README.md`](README.md).
